## Инициализация

In [ ]:
import copy
import json
import os
import random

import albumentations as A
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import torchvision
import torchvision.transforms.v2 as tfs
from albumentations.pytorch import ToTensorV2
from torchinfo import summary
from torchvision.datasets import ImageFolder
from tqdm import tqdm

random.seed(0)
np.random.seed(0)
torch.manual_seed(0)
torch.cuda.manual_seed(0)
torch.backends.cudnn.deterministic = True

In [ ]:
dataset_label = 'dataset'
train_label = 'train'
test_label = 'test'
format_label = 'format.json'

train_directory = os.path.join(dataset_label, train_label)
test_directory = os.path.join(dataset_label, test_label)
format_directory = os.path.join(dataset_label, format_label)

In [ ]:
train_count = 0
for _, _, files in os.walk(train_directory):
    train_count += len(files)

test_count = 0
for _, _, files in os.walk(test_directory):
    test_count += len(files)

In [ ]:
with open(format_directory) as file:
    classes = json.load(file)

classes = list(classes.values())

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используется устройство: {device}")

## Базовые функции и классы

In [ ]:
class AlbumentationsTransform:
    def __init__(self, albumentations_transform):
        self.albumentations_transform = albumentations_transform

    def __call__(self, img):
        img = np.array(img)
        augmented = self.albumentations_transform(image=img)
        return augmented['image']

In [ ]:
def loss_model(model, loss_func, data):
  model.eval()
  Q = 0
  count = 0

  for x, y in data:
    x = x.to(device)
    y = y.to(device)

    with torch.no_grad():
      p = model(x)
      loss = loss_func(p, y)
      Q += loss.item()
      count += 1

  Q /= count

  return Q

def accuracy_model(model, device, data):
  model.eval()
  Q = 0

  for x, y in data:
    x = x.to(device)
    y = y.to(device)

    with torch.no_grad():
      p = model(x)
      p = torch.argmax(p, dim=1)
      Q += torch.sum(p == y).item()

  count = len(data.dataset)

  Q /= count
  
  return Q

In [ ]:
def train_model(model, device, train_data, train_data_val, optimizer, loss_func, epochs):
  model.to(device)
  acc_lst_val = []
  acc_lst = []
  for _e in range(epochs):
    model.train()
    loss_mean = 0
    lm_count = 0
    train_tqdm = tqdm(train_data, leave=False)

    # Обучение на 1 батче
    for x_train, y_train in train_tqdm:
      x_train = x_train.to(device)
      y_train = y_train.to(device)

      predict = model(x_train)
      loss = loss_func(predict, y_train)

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

      # обновление среднего по формуле
      lm_count += 1
      loss_mean = 1 / lm_count * loss.item() + (1 - 1 / lm_count) * loss_mean
      train_tqdm.set_description(f"Epoch [{_e+1}/{epochs}], loss: {loss_mean:.4f}")

    # валидация
    # Q_val = loss_model(model, loss_func, train_data_val)

    train_acc = accuracy_model(model, device, train_data)
    val_acc = accuracy_model(model, device, train_data_val)

    acc_lst.append(train_acc)
    acc_lst_val.append(val_acc)

    print(f'Epoch [{_e+1}/{epochs}] | acc_train={train_acc:.3f}, acc_val={val_acc:.3f}')
  return model, acc_lst, acc_lst_val

In [ ]:
def show_loss(loss_lst, loss_lst_val):
  plt.plot(loss_lst, label='predict')
  plt.plot(loss_lst_val, label='real')
  plt.legend()
  plt.grid()
  plt.show()

In [ ]:
def split_data(transform, train_size=0.8, batch_size=100, album=False):
    if album:
        transform = AlbumentationsTransform(transform)
    dataset_train = ImageFolder(train_directory, transform=transform)

    train_size = int(train_size * len(dataset_train))
    val_size = len(dataset_train) - train_size
    d_train, d_val = data.random_split(dataset_train, [train_size, val_size])

    train_data = data.DataLoader(d_train, batch_size=batch_size, shuffle=True)
    train_data_val = data.DataLoader(d_val, batch_size=batch_size, shuffle=False)

    d_test = ImageFolder(test_directory, transform=transform)
    test_data = data.DataLoader(d_test, batch_size=batch_size, shuffle=False)

    return train_data, train_data_val, test_data

## Пользовательская нейронная сеть

### Начальная модель

In [ ]:
class ModelNN(nn.Module):
  def __init__(self):
    super().__init__()
    self.layers = nn.Sequential(
        torch.nn.Conv2d(
            in_channels=3, out_channels=6, kernel_size=5, padding=2), # (batch, 6, 200, 200)
        torch.nn.ReLU(),
        torch.nn.AvgPool2d(kernel_size=2, stride=2), # (batch, 6, 100, 100)

        torch.nn.Conv2d(
            in_channels=6, out_channels=16, kernel_size=5, padding=2), # (batch, 16, 100, 100)
        torch.nn.ReLU(),
        torch.nn.AvgPool2d(kernel_size=2, stride=2), # (batch, 16, 50, 50)

        torch.nn.Conv2d(
            in_channels=16, out_channels=16, kernel_size=5, padding=0), # (batch, 16, 46, 46)
        torch.nn.ReLU(),
        torch.nn.AvgPool2d(kernel_size=2, stride=2), # (batch, 16, 23, 23)

        nn.Flatten(),

        torch.nn.Linear(23 * 23 * 16, 120),
        torch.nn.ReLU(),
        torch.nn.Linear(120, 84),
        torch.nn.ReLU(),
        torch.nn.Linear(84, len(classes))
    )

  def forward(self, x):
    return self.layers(x)

In [ ]:
model = ModelNN()
model = model.to(device)
sum(p.numel() for p in model.parameters())

In [ ]:
transforms = tfs.Compose(
    [
        tfs.ToImage(),
        tfs.ToDtype(torch.float32, scale=True)
    ]
)

train_size = 0.8
batch_size = 100
album = False

In [ ]:
train_data, train_data_val, test_data = split_data(transforms, train_size=train_size, 
                                                   batch_size=batch_size, album=album)

In [ ]:
optimizer = optim.Adam(params=model.parameters(), lr=0.001)
loss_func = nn.CrossEntropyLoss()
epochs = 20

In [ ]:
model, res_lst, res_lst_val = train_model(model, device, train_data, train_data_val, 
                                            optimizer, loss_func, epochs)

In [ ]:
print(accuracy_model(model, device, test_data))
show_loss(res_lst, res_lst_val)

### Разное кол-во слоев

#### Больше сверточных слоев

In [ ]:
class ModelNN_cnn5(nn.Module):
  def __init__(self):
    super().__init__()
    self.layers = nn.Sequential(
      torch.nn.Conv2d(in_channels=3, out_channels=6, kernel_size=5, padding=2), # (batch, 6, 200, 200)
      torch.nn.ReLU(),
      torch.nn.AvgPool2d(kernel_size=2, stride=2),                             # (batch, 6, 100, 100)

      torch.nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5, padding=2), # (batch, 16, 100, 100)
      torch.nn.ReLU(),
      torch.nn.AvgPool2d(kernel_size=2, stride=2),                              # (batch, 16, 50, 50)

      torch.nn.Conv2d(in_channels=16, out_channels=16, kernel_size=5, padding=0), # (batch, 16, 46, 46)
      torch.nn.ReLU(),
      torch.nn.AvgPool2d(kernel_size=2, stride=2),                               # (batch, 16, 23, 23)

      # Новый сверточный слой 1
      torch.nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1), # (batch, 32, 23, 23)
      torch.nn.ReLU(),
      torch.nn.AvgPool2d(kernel_size=2, stride=2),                               # (batch, 32, 11, 11)

      # Новый сверточный слой 2
      torch.nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1), # (batch, 64, 11, 11)
      torch.nn.ReLU(),
      torch.nn.AvgPool2d(kernel_size=2, stride=2),                               # (batch, 64, 5, 5)

      nn.Flatten(),

      torch.nn.Linear(5 * 5 * 64, 120),
      torch.nn.ReLU(),
      torch.nn.Linear(120, 84),
      torch.nn.ReLU(),
      torch.nn.Linear(84, len(classes))
    )

  def forward(self, x):
    return self.layers(x)


Мы увеличили кол-во сверточных слоев, однако кол-во параметров у модели уменьшилось в 5 раз

In [ ]:
model = ModelNN_cnn5()
model = model.to(device)
sum(p.numel() for p in model.parameters())

In [ ]:
optimizer = optim.Adam(params=model.parameters(), lr=0.001)
loss_func = nn.CrossEntropyLoss()
epochs = 20

In [ ]:
model, res_lst, res_lst_val = train_model(model, device, train_data, train_data_val, 
                                            optimizer, loss_func, epochs)

In [ ]:
print(accuracy_model(model, device, test_data))
show_loss(res_lst, res_lst_val)

Мы видим, что, несмотря на меньшее кол-во параметров, модель с большим кол-во сверточных слоев справилась с задачей лучше. Можно предположить, что начальная модель также может дойти до такой же точности, т.к. по графику можно заметить, что модель не дошла до плато/переобучения.

Из эксперимента можно утверждать: увеличение количества свёрточных слоев позволяет модели находить больее сложные и абстрактные признаки, что увеличивает ее точность.

#### Больше линейных слоев

In [ ]:
class ModelNN_fc5(nn.Module):
  def __init__(self):
    super().__init__()
    self.layers = nn.Sequential(
      torch.nn.Conv2d(in_channels=3, out_channels=6, kernel_size=5, padding=2),  # (batch, 6, 200, 200)
      torch.nn.ReLU(),
      torch.nn.AvgPool2d(kernel_size=2, stride=2),  # (batch, 6, 100, 100)

      torch.nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5, padding=2),  # (batch, 16, 100, 100)
      torch.nn.ReLU(),
      torch.nn.AvgPool2d(kernel_size=2, stride=2),  # (batch, 16, 50, 50)

      torch.nn.Conv2d(in_channels=16, out_channels=16, kernel_size=5, padding=0),  # (batch, 16, 46, 46)
      torch.nn.ReLU(),
      torch.nn.AvgPool2d(kernel_size=2, stride=2),  # (batch, 16, 23, 23)

      nn.Flatten(),

      torch.nn.Linear(23 * 23 * 16, 120),
      torch.nn.ReLU(),

      # Дополнительные линейные слои
      torch.nn.Linear(120, 100),
      torch.nn.ReLU(),
      torch.nn.Linear(100, 90),
      torch.nn.ReLU(),

      torch.nn.Linear(90, 84),
      torch.nn.ReLU(),

      torch.nn.Linear(84, 50),
      torch.nn.ReLU(),

      torch.nn.Linear(50, len(classes))
    )

  def forward(self, x):
    return self.layers(x)


Кол-во параметров практически неизменилось

In [ ]:
model = ModelNN_fc5()
model = model.to(device)
sum(p.numel() for p in model.parameters())

In [ ]:
optimizer = optim.Adam(params=model.parameters(), lr=0.001)
loss_func = nn.CrossEntropyLoss()
epochs = 20

В данной модели линейный классификатор является достаточно сложных по структуре, однако тот на вход получает небольшое кол-во признаков, из-за чего модель "путается" и точность падает.

Вывод из эксперимента: для задач подобного масштаба, линейный классификатор должен быть небольшим.

In [ ]:
model, res_lst, res_lst_val = train_model(model, device, train_data, train_data_val, 
                                            optimizer, loss_func, epochs)

In [ ]:
print(accuracy_model(model, device, test_data))
show_loss(res_lst, res_lst_val)

### Число карт признаков в слоях

#### Больше

In [ ]:
class ModelNN_maps64(nn.Module):
  def __init__(self):
    super().__init__()
    self.layers = nn.Sequential(
      torch.nn.Conv2d(
          in_channels=3, out_channels=16, kernel_size=5, padding=2),  # (batch, 16, 200, 200)
      torch.nn.ReLU(),
      torch.nn.AvgPool2d(kernel_size=2, stride=2),                   # (batch, 16, 100, 100)

      torch.nn.Conv2d(
          in_channels=16, out_channels=48, kernel_size=5, padding=2),  # (batch, 48, 100, 100)
      torch.nn.ReLU(),
      torch.nn.AvgPool2d(kernel_size=2, stride=2),                    # (batch, 48, 50, 50)

      torch.nn.Conv2d(
          in_channels=48, out_channels=64, kernel_size=5, padding=0),  # (batch, 64, 46, 46)
      torch.nn.ReLU(),
      torch.nn.AvgPool2d(kernel_size=2, stride=2),                    # (batch, 64, 23, 23)

      nn.Flatten(),

      torch.nn.Linear(23 * 23 * 64, 120),  # изменился размер входа
      torch.nn.ReLU(),
      torch.nn.Linear(120, 84),
      torch.nn.ReLU(),
      torch.nn.Linear(84, len(classes))
    )

  def forward(self, x):
    return self.layers(x)


In [ ]:
model = ModelNN_maps64()
model = model.to(device)
sum(p.numel() for p in model.parameters())

In [ ]:
optimizer = optim.Adam(params=model.parameters(), lr=0.001)
loss_func = nn.CrossEntropyLoss()
epochs = 20

Модель имеет идентичную точность, что и начальная модель, однако имеет в 4 раза больше параметров.

Также присутствуют признаки начала переобучения: ошибка на тренировочных данных значительно выше чем на валидационных. Похожая ситуация была и у начальной модели, на разница была не столь существенной.

In [ ]:
model, res_lst, res_lst_val = train_model(model, device, train_data, train_data_val, 
                                            optimizer, loss_func, epochs)

In [ ]:
print(accuracy_model(model, device, test_data))
show_loss(res_lst, res_lst_val)

#### Меньше

In [ ]:
class ModelNN_maps8(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            torch.nn.Conv2d(
                in_channels=3, out_channels=3, kernel_size=5, padding=2),  # (batch, 3, 200, 200)
            torch.nn.ReLU(),
            torch.nn.AvgPool2d(kernel_size=2, stride=2),                  # (batch, 3, 100, 100)

            torch.nn.Conv2d(
                in_channels=3, out_channels=8, kernel_size=5, padding=2),  # (batch, 8, 100, 100)
            torch.nn.ReLU(),
            torch.nn.AvgPool2d(kernel_size=2, stride=2),                  # (batch, 8, 50, 50)

            torch.nn.Conv2d(
                in_channels=8, out_channels=8, kernel_size=5, padding=0),  # (batch, 8, 46, 46)
            torch.nn.ReLU(),
            torch.nn.AvgPool2d(kernel_size=2, stride=2),                  # (batch, 8, 23, 23)

            nn.Flatten(),

            torch.nn.Linear(23 * 23 * 8, 120),  # размер входа изменился
            torch.nn.ReLU(),
            torch.nn.Linear(120, 84),
            torch.nn.ReLU(),
            torch.nn.Linear(84, len(classes))
        )

    def forward(self, x):
        return self.layers(x)


In [ ]:
model = ModelNN_maps8()
model = model.to(device)
sum(p.numel() for p in model.parameters())

In [ ]:
optimizer = optim.Adam(params=model.parameters(), lr=0.001)
loss_func = nn.CrossEntropyLoss()
epochs = 20

In [ ]:
model, res_lst, res_lst_val = train_model(model, device, train_data, train_data_val, 
                                            optimizer, loss_func, epochs)

In [ ]:
print(accuracy_model(model, device, test_data))
show_loss(res_lst, res_lst_val)

### Функции активации

#### LeakyReLU

In [ ]:
class ModelNN_lrelu(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            torch.nn.Conv2d(
                in_channels=3, out_channels=3, kernel_size=5, padding=2),  # (batch, 3, 200, 200)
            torch.nn.LeakyReLU(),
            torch.nn.AvgPool2d(kernel_size=2, stride=2),                  # (batch, 3, 100, 100)

            torch.nn.Conv2d(
                in_channels=3, out_channels=8, kernel_size=5, padding=2),  # (batch, 8, 100, 100)
            torch.nn.LeakyReLU(),
            torch.nn.AvgPool2d(kernel_size=2, stride=2),                  # (batch, 8, 50, 50)

            torch.nn.Conv2d(
                in_channels=8, out_channels=8, kernel_size=5, padding=0),  # (batch, 8, 46, 46)
            torch.nn.LeakyReLU(),
            torch.nn.AvgPool2d(kernel_size=2, stride=2),                  # (batch, 8, 23, 23)

            nn.Flatten(),

            torch.nn.Linear(23 * 23 * 8, 120),  # размер входа изменился
            torch.nn.LeakyReLU(),
            torch.nn.Linear(120, 84),
            torch.nn.LeakyReLU(),
            torch.nn.Linear(84, len(classes))
        )

    def forward(self, x):
        return self.layers(x)


In [ ]:
model = ModelNN_lrelu()
model = model.to(device)
sum(p.numel() for p in model.parameters())

In [ ]:
optimizer = optim.Adam(params=model.parameters(), lr=0.001)
loss_func = nn.CrossEntropyLoss()
epochs = 20

In [ ]:
model, res_lst, res_lst_val = train_model(model, device, train_data, train_data_val, 
                                            optimizer, loss_func, epochs)

In [ ]:
print(accuracy_model(model, device, test_data))
show_loss(res_lst, res_lst_val)

#### tanh

In [ ]:
class ModelNN_tanh(nn.Module):
  def __init__(self):
    super().__init__()
    self.layers = nn.Sequential(
        torch.nn.Conv2d(
            in_channels=3, out_channels=6, kernel_size=5, padding=2), # (batch, 6, 200, 200)
        torch.nn.Tanh(),
        torch.nn.AvgPool2d(kernel_size=2, stride=2), # (batch, 6, 100, 100)

        torch.nn.Conv2d(
            in_channels=6, out_channels=16, kernel_size=5, padding=2), # (batch, 16, 100, 100)
        torch.nn.Tanh(),
        torch.nn.AvgPool2d(kernel_size=2, stride=2), # (batch, 16, 50, 50)

        torch.nn.Conv2d(
            in_channels=16, out_channels=16, kernel_size=5, padding=0), # (batch, 16, 46, 46)
        torch.nn.Tanh(),
        torch.nn.AvgPool2d(kernel_size=2, stride=2), # (batch, 16, 23, 23)

        nn.Flatten(),

        torch.nn.Linear(23 * 23 * 16, 120),
        torch.nn.Tanh(),
        torch.nn.Linear(120, 84),
        torch.nn.Tanh(),
        torch.nn.Linear(84, len(classes))
    )

  def forward(self, x):
    return self.layers(x)

In [ ]:
model = ModelNN_tanh()
model = model.to(device)
sum(p.numel() for p in model.parameters())

In [ ]:
optimizer = optim.Adam(params=model.parameters(), lr=0.001)
loss_func = nn.CrossEntropyLoss()
epochs = 20

In [ ]:
model, res_lst, res_lst_val = train_model(model, device, train_data, train_data_val, 
                                            optimizer, loss_func, epochs)

In [ ]:
print(accuracy_model(model, device, test_data))
show_loss(res_lst, res_lst_val)

### Размер ядра свертки

#### kernel_size=3

In [ ]:
class ModelNN_ks3(nn.Module):
  def __init__(self):
    super().__init__()
    self.layers = nn.Sequential(
        torch.nn.Conv2d(
            in_channels=3, out_channels=6, kernel_size=3, padding=1), # (batch, 6, 200, 200)
        torch.nn.ReLU(),
        torch.nn.AvgPool2d(kernel_size=2, stride=2), # (batch, 6, 100, 100)

        torch.nn.Conv2d(
            in_channels=6, out_channels=16, kernel_size=3, padding=1), # (batch, 16, 100, 100)
        torch.nn.ReLU(),
        torch.nn.AvgPool2d(kernel_size=2, stride=2), # (batch, 16, 50, 50)

        torch.nn.Conv2d(
            in_channels=16, out_channels=16, kernel_size=3, padding=0), # (batch, 16, 46, 46)
        torch.nn.ReLU(),
        torch.nn.AvgPool2d(kernel_size=2, stride=2), # (batch, 16, 23, 23)

        nn.Flatten(),

        torch.nn.Linear(24 * 24 * 16, 120),
        torch.nn.ReLU(),
        torch.nn.Linear(120, 84),
        torch.nn.ReLU(),
        torch.nn.Linear(84, len(classes))
    )

  def forward(self, x):
    return self.layers(x)

In [ ]:
model = ModelNN_ks3()
model = model.to(device)
sum(p.numel() for p in model.parameters())

In [ ]:
optimizer = optim.Adam(params=model.parameters(), lr=0.001)
loss_func = nn.CrossEntropyLoss()
epochs = 20

In [ ]:
model, res_lst, res_lst_val = train_model(model, device, train_data, train_data_val, 
                                            optimizer, loss_func, epochs)

In [ ]:
print(accuracy_model(model, device, test_data))
show_loss(res_lst, res_lst_val)

#### kernel_size=7

In [ ]:
class ModelNN_ks7(nn.Module):
  def __init__(self):
    super().__init__()
    self.layers = nn.Sequential(
        torch.nn.Conv2d(
            in_channels=3, out_channels=6, kernel_size=7, padding=2), # (batch, 6, 200, 200)
        torch.nn.ReLU(),
        torch.nn.AvgPool2d(kernel_size=2, stride=2), # (batch, 6, 100, 100)

        torch.nn.Conv2d(
            in_channels=6, out_channels=16, kernel_size=7, padding=2), # (batch, 16, 100, 100)
        torch.nn.ReLU(),
        torch.nn.AvgPool2d(kernel_size=2, stride=2), # (batch, 16, 50, 50)

        torch.nn.Conv2d(
            in_channels=16, out_channels=16, kernel_size=7, padding=0), # (batch, 16, 46, 46)
        torch.nn.ReLU(),
        torch.nn.AvgPool2d(kernel_size=2, stride=2), # (batch, 16, 23, 23)

        nn.Flatten(),

        torch.nn.Linear(21 * 21 * 16, 120),
        torch.nn.ReLU(),
        torch.nn.Linear(120, 84),
        torch.nn.ReLU(),
        torch.nn.Linear(84, len(classes))
    )

  def forward(self, x):
    return self.layers(x)

In [ ]:
model = ModelNN_ks7()
model = model.to(device)
sum(p.numel() for p in model.parameters())

In [ ]:
optimizer = optim.Adam(params=model.parameters(), lr=0.001)
loss_func = nn.CrossEntropyLoss()
epochs = 20

In [ ]:
model, res_lst, res_lst_val = train_model(model, device, train_data, train_data_val, 
                                            optimizer, loss_func, epochs)

In [ ]:
print(accuracy_model(model, device, test_data))
show_loss(res_lst, res_lst_val)

### Padding

#### padding=3

In [ ]:
class ModelNN_p3(nn.Module):
  def __init__(self):
    super().__init__()
    self.layers = nn.Sequential(
        torch.nn.Conv2d(
            in_channels=3, out_channels=6, kernel_size=5, padding=3), # (batch, 6, 202, 202)
        torch.nn.ReLU(),
        torch.nn.AvgPool2d(kernel_size=2, stride=2), # (batch, 6, 101, 101)

        torch.nn.Conv2d(
            in_channels=6, out_channels=16, kernel_size=5, padding=3), # (batch, 16, 103, 103)
        torch.nn.ReLU(),
        torch.nn.AvgPool2d(kernel_size=2, stride=2), # (batch, 16, 51, 51)

        torch.nn.Conv2d(
            in_channels=16, out_channels=16, kernel_size=5, padding=3), # (batch, 16, 53, 53)
        torch.nn.ReLU(),
        torch.nn.AvgPool2d(kernel_size=2, stride=2), # (batch, 16, 26, 26)

        nn.Flatten(),

        torch.nn.Linear(26 * 26 * 16, 120),
        torch.nn.ReLU(),
        torch.nn.Linear(120, 84),
        torch.nn.ReLU(),
        torch.nn.Linear(84, len(classes))
    )

  def forward(self, x):
    return self.layers(x)

In [ ]:
model = ModelNN_p3()
model = model.to(device)
sum(p.numel() for p in model.parameters())

In [ ]:
optimizer = optim.Adam(params=model.parameters(), lr=0.001)
loss_func = nn.CrossEntropyLoss()
epochs = 20

In [ ]:
model, res_lst, res_lst_val = train_model(model, device, train_data, train_data_val, 
                                            optimizer, loss_func, epochs)

In [ ]:
print(accuracy_model(model, device, test_data))
show_loss(res_lst, res_lst_val)

#### padding=0

In [ ]:
class ModelNN_p0(nn.Module):
  def __init__(self):
    super().__init__()
    self.layers = nn.Sequential(
        torch.nn.Conv2d(
            in_channels=3, out_channels=6, kernel_size=5, padding=0), # (batch, 6, 196, 196)
        torch.nn.ReLU(),
        torch.nn.AvgPool2d(kernel_size=2, stride=2), # (batch, 6, 98, 98)

        torch.nn.Conv2d(
            in_channels=6, out_channels=16, kernel_size=5, padding=0), # (batch, 16, 94, 94)
        torch.nn.ReLU(),
        torch.nn.AvgPool2d(kernel_size=2, stride=2), # (batch, 16, 47, 47)

        torch.nn.Conv2d(
            in_channels=16, out_channels=16, kernel_size=5, padding=0), # (batch, 16, 43, 43)
        torch.nn.ReLU(),
        torch.nn.AvgPool2d(kernel_size=2, stride=2), # (batch, 16, 21, 21)

        nn.Flatten(),

        torch.nn.Linear(21 * 21 * 16, 120),
        torch.nn.ReLU(),
        torch.nn.Linear(120, 84),
        torch.nn.ReLU(),
        torch.nn.Linear(84, len(classes))
    )

  def forward(self, x):
    return self.layers(x)

In [ ]:
model = ModelNN_p0()
model = model.to(device)
sum(p.numel() for p in model.parameters())

In [ ]:
optimizer = optim.Adam(params=model.parameters(), lr=0.001)
loss_func = nn.CrossEntropyLoss()
epochs = 20

In [ ]:
model, res_lst, res_lst_val = train_model(model, device, train_data, train_data_val, 
                                            optimizer, loss_func, epochs)

In [ ]:
print(accuracy_model(model, device, test_data))
show_loss(res_lst, res_lst_val)

### Stride

#### stride=3

In [ ]:
class ModelNN_s3(nn.Module):
  def __init__(self):
    super().__init__()
    self.layers = nn.Sequential(
        torch.nn.Conv2d(
            in_channels=3, out_channels=6, kernel_size=5, padding=2), # (batch, 6, 200, 200)
        torch.nn.ReLU(),
        torch.nn.AvgPool2d(kernel_size=2, stride=3), # (batch, 6, 66, 66)

        torch.nn.Conv2d(
            in_channels=6, out_channels=16, kernel_size=5, padding=2), # (batch, 16, 66, 66)
        torch.nn.ReLU(),
        torch.nn.AvgPool2d(kernel_size=2, stride=3), # (batch, 16, 22, 22)

        torch.nn.Conv2d(
            in_channels=16, out_channels=16, kernel_size=5, padding=0), # (batch, 16, 18, 18)
        torch.nn.ReLU(),
        torch.nn.AvgPool2d(kernel_size=2, stride=3), # (batch, 16, 6, 6)

        nn.Flatten(),

        torch.nn.Linear(6 * 6 * 16, 120),
        torch.nn.ReLU(),
        torch.nn.Linear(120, 84),
        torch.nn.ReLU(),
        torch.nn.Linear(84, len(classes))
    )

  def forward(self, x):
    return self.layers(x)

In [ ]:
model = ModelNN_s3()
model = model.to(device)
sum(p.numel() for p in model.parameters())

In [ ]:
optimizer = optim.Adam(params=model.parameters(), lr=0.001)
loss_func = nn.CrossEntropyLoss()
epochs = 20

In [ ]:
model, res_lst, res_lst_val = train_model(model, device, train_data, train_data_val, 
                                            optimizer, loss_func, epochs)

In [ ]:
print(accuracy_model(model, device, test_data))
show_loss(res_lst, res_lst_val)

#### stride=3-3

In [ ]:
class ModelNN_s3_3(nn.Module):
  def __init__(self):
    super().__init__()
    self.layers = nn.Sequential(
        torch.nn.Conv2d(
            in_channels=3, out_channels=6, kernel_size=5, padding=2), # (batch, 6, 200, 200)
        torch.nn.ReLU(),
        torch.nn.AvgPool2d(kernel_size=3, stride=3), # (batch, 6, 66, 66)

        torch.nn.Conv2d(
            in_channels=6, out_channels=16, kernel_size=5, padding=2), # (batch, 16, 66, 66)
        torch.nn.ReLU(),
        torch.nn.AvgPool2d(kernel_size=3, stride=3), # (batch, 16, 22, 22)

        torch.nn.Conv2d(
            in_channels=16, out_channels=16, kernel_size=5, padding=0), # (batch, 16, 18, 18)
        torch.nn.ReLU(),
        torch.nn.AvgPool2d(kernel_size=3, stride=3), # (batch, 16, 6, 6)

        nn.Flatten(),

        torch.nn.Linear(6 * 6 * 16, 120),
        torch.nn.ReLU(),
        torch.nn.Linear(120, 84),
        torch.nn.ReLU(),
        torch.nn.Linear(84, len(classes))
    )

  def forward(self, x):
    return self.layers(x)

In [ ]:
model = ModelNN_s3_3()
model = model.to(device)
sum(p.numel() for p in model.parameters())

In [ ]:
optimizer = optim.Adam(params=model.parameters(), lr=0.001)
loss_func = nn.CrossEntropyLoss()
epochs = 20

In [ ]:
model, res_lst, res_lst_val = train_model(model, device, train_data, train_data_val, 
                                            optimizer, loss_func, epochs)

In [ ]:
print(accuracy_model(model, device, test_data))
show_loss(res_lst, res_lst_val)

### Функция пуллинга

#### AdaptiveMaxPool2d

In [ ]:
class ModelNN_ad_maxpool(nn.Module):
  def __init__(self):
    super().__init__()
    self.layers = nn.Sequential(
        torch.nn.Conv2d(
            in_channels=3, out_channels=6, kernel_size=5, padding=2), # (batch, 6, 200, 200)
        torch.nn.ReLU(),
        torch.nn.AdaptiveMaxPool2d((150, 150)), # (batch, 6, 150, 150)

        torch.nn.Conv2d(
            in_channels=6, out_channels=16, kernel_size=5, padding=2), # (batch, 16, 100, 100)
        torch.nn.ReLU(),
        torch.nn.AdaptiveMaxPool2d((50, 50)), # (batch, 16, 50, 50)

        torch.nn.Conv2d(
            in_channels=16, out_channels=16, kernel_size=5, padding=0), # (batch, 16, 46, 46)
        torch.nn.ReLU(),
        torch.nn.AdaptiveMaxPool2d((15, 15)), # (batch, 16, 23, 23)

        nn.Flatten(),

        torch.nn.Linear(15 * 15 * 16, 120),
        torch.nn.ReLU(),
        torch.nn.Linear(120, 84),
        torch.nn.ReLU(),
        torch.nn.Linear(84, len(classes))
    )

  def forward(self, x):
    return self.layers(x)

In [ ]:
model = ModelNN_ad_maxpool()
model = model.to(device)
sum(p.numel() for p in model.parameters())

In [ ]:
optimizer = optim.Adam(params=model.parameters(), lr=0.001)
loss_func = nn.CrossEntropyLoss()
epochs = 20

In [ ]:
model, res_lst, res_lst_val = train_model(model, device, train_data, train_data_val, 
                                            optimizer, loss_func, epochs)

In [ ]:
print(accuracy_model(model, device, test_data))
show_loss(res_lst, res_lst_val)

#### MaxPool2d

In [ ]:
class ModelNN_maxpool(nn.Module):
  def __init__(self):
    super().__init__()
    self.layers = nn.Sequential(
        torch.nn.Conv2d(
            in_channels=3, out_channels=6, kernel_size=5, padding=2), # (batch, 6, 200, 200)
        torch.nn.ReLU(),
        torch.nn.MaxPool2d(kernel_size=2, stride=2), # (batch, 6, 100, 100)

        torch.nn.Conv2d(
            in_channels=6, out_channels=16, kernel_size=5, padding=2), # (batch, 16, 100, 100)
        torch.nn.ReLU(),
        torch.nn.MaxPool2d(kernel_size=2, stride=2), # (batch, 16, 50, 50)

        torch.nn.Conv2d(
            in_channels=16, out_channels=16, kernel_size=5, padding=0), # (batch, 16, 46, 46)
        torch.nn.ReLU(),
        torch.nn.MaxPool2d(kernel_size=2, stride=2), # (batch, 16, 23, 23)

        nn.Flatten(),

        torch.nn.Linear(23 * 23 * 16, 120),
        torch.nn.ReLU(),
        torch.nn.Linear(120, 84),
        torch.nn.ReLU(),
        torch.nn.Linear(84, len(classes))
    )

  def forward(self, x):
    return self.layers(x)

In [ ]:
model = ModelNN_maxpool()
model = model.to(device)
sum(p.numel() for p in model.parameters())

In [ ]:
optimizer = optim.Adam(params=model.parameters(), lr=0.001)
loss_func = nn.CrossEntropyLoss()
epochs = 20

In [ ]:
model, res_lst, res_lst_val = train_model(model, device, train_data, train_data_val, 
                                            optimizer, loss_func, epochs)

In [ ]:
print(accuracy_model(model, device, test_data))
show_loss(res_lst, res_lst_val)

### Оптимизатор

#### RMSprop

In [ ]:
model = ModelNN()
model = model.to(device)
sum(p.numel() for p in model.parameters())

In [ ]:
optimizer = optim.RMSprop(params=model.parameters(), lr=0.001)
loss_func = nn.CrossEntropyLoss()
epochs = 20

In [ ]:
model, res_lst, res_lst_val = train_model(model, device, train_data, train_data_val, 
                                            optimizer, loss_func, epochs)

In [ ]:
print(accuracy_model(model, device, test_data))
show_loss(res_lst, res_lst_val)

#### SGD

In [ ]:
model = ModelNN()
model = model.to(device)
sum(p.numel() for p in model.parameters())

In [ ]:
optimizer = optim.SGD(params=model.parameters(), lr=0.001, momentum=0.9)
loss_func = nn.CrossEntropyLoss()
epochs = 20

In [ ]:
model, res_lst, res_lst_val = train_model(model, device, train_data, train_data_val, 
                                            optimizer, loss_func, epochs)

In [ ]:
print(accuracy_model(model, device, test_data))
show_loss(res_lst, res_lst_val)

### Скорость обучения

#### lr=0.01

In [ ]:
model = ModelNN()
model = model.to(device)
sum(p.numel() for p in model.parameters())

In [ ]:
optimizer = optim.Adam(params=model.parameters(), lr=0.01)
loss_func = nn.CrossEntropyLoss()
epochs = 20

In [ ]:
model, res_lst, res_lst_val = train_model(model, device, train_data, train_data_val, 
                                            optimizer, loss_func, epochs)

In [ ]:
print(accuracy_model(model, device, test_data))
show_loss(res_lst, res_lst_val)

#### lr=0.005

In [ ]:
model = ModelNN()
model = model.to(device)
sum(p.numel() for p in model.parameters())

In [ ]:
optimizer = optim.Adam(params=model.parameters(), lr=0.005)
loss_func = nn.CrossEntropyLoss()
epochs = 20

In [ ]:
model, res_lst, res_lst_val = train_model(model, device, train_data, train_data_val, 
                                            optimizer, loss_func, epochs)

In [ ]:
print(accuracy_model(model, device, test_data))
show_loss(res_lst, res_lst_val)

### Размер батча

#### batch_size = 150

In [ ]:
model = ModelNN()
model = model.to(device)
sum(p.numel() for p in model.parameters())

In [ ]:
transforms = tfs.Compose(
    [
        tfs.ToImage(),
        tfs.ToDtype(torch.float32, scale=True)
    ]
)

train_size = 0.8
batch_size = 150
album = False

In [ ]:
train_data, train_data_val, test_data = split_data(transforms, train_size=train_size, 
                                                   batch_size=batch_size, album=album)

In [ ]:
optimizer = optim.Adam(params=model.parameters(), lr=0.001)
loss_func = nn.CrossEntropyLoss()
epochs = 20

In [ ]:
model, res_lst, res_lst_val = train_model(model, device, train_data, train_data_val, 
                                            optimizer, loss_func, epochs)

In [ ]:
print(accuracy_model(model, device, test_data))
show_loss(res_lst, res_lst_val)

#### batch_size = 50

In [ ]:
model = ModelNN()
model = model.to(device)
sum(p.numel() for p in model.parameters())

In [ ]:
transforms = tfs.Compose(
    [
        tfs.ToImage(),
        tfs.ToDtype(torch.float32, scale=True)
    ]
)

train_size = 0.8
batch_size = 50
album = False

In [ ]:
train_data, train_data_val, test_data = split_data(transforms, train_size=train_size, 
                                                   batch_size=batch_size, album=album)

In [ ]:
optimizer = optim.Adam(params=model.parameters(), lr=0.001)
loss_func = nn.CrossEntropyLoss()
epochs = 20

In [ ]:
model, res_lst, res_lst_val = train_model(model, device, train_data, train_data_val, 
                                            optimizer, loss_func, epochs)

In [ ]:
print(accuracy_model(model, device, test_data))
show_loss(res_lst, res_lst_val)

### Число эпох

#### epochs = 30

In [ ]:
model = ModelNN()
model = model.to(device)
sum(p.numel() for p in model.parameters())

In [ ]:
optimizer = optim.Adam(params=model.parameters(), lr=0.001)
loss_func = nn.CrossEntropyLoss()
epochs = 30

In [ ]:
transforms = tfs.Compose(
    [
        tfs.ToImage(),
        tfs.ToDtype(torch.float32, scale=True)
    ]
)

train_size = 0.8
batch_size = 100
album = False

In [ ]:
train_data, train_data_val, test_data = split_data(transforms, train_size=train_size, 
                                                   batch_size=batch_size, album=album)

In [ ]:
model, res_lst, res_lst_val = train_model(model, device, train_data, train_data_val, 
                                            optimizer, loss_func, epochs)

In [ ]:
print(accuracy_model(model, device, test_data))
show_loss(res_lst, res_lst_val)

### Модель, с учетом экспериментов

#### Новая модель

In [ ]:
class ModelNN_v2(nn.Module):
  def __init__(self):
    super().__init__()
    self.layers = nn.Sequential(
      torch.nn.Conv2d(in_channels=3, out_channels=6, kernel_size=5, padding=2), # (batch, 6, 200, 200)
      torch.nn.ReLU(),
      torch.nn.AvgPool2d(kernel_size=3, stride=3),                              # (batch, 6, 66, 66)

      torch.nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5, padding=2), # (batch, 16, 66, 66)
      torch.nn.ReLU(),
      torch.nn.AvgPool2d(kernel_size=3, stride=3),                               # (batch, 16, 22, 22)

      torch.nn.Conv2d(in_channels=16, out_channels=16, kernel_size=5, padding=2), # (batch, 16, 22, 22)
      torch.nn.ReLU(),
      torch.nn.AvgPool2d(kernel_size=3, stride=2),                                # (batch, 16, 7, 7)

      # Новый сверточный слой 1
      torch.nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1), # (batch, 32, 7, 7)
      torch.nn.ReLU(),
      torch.nn.AvgPool2d(kernel_size=3, stride=1),                                # (batch, 32, 11, 11)

      # Новый сверточный слой 2
      torch.nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1), # (batch, 64, 11, 11)
      torch.nn.ReLU(),
      torch.nn.AvgPool2d(kernel_size=3, stride=1),                                # (batch, 64, 5, 5)

      nn.Flatten(),

      torch.nn.Linear(6 * 6 * 64, 120),
      torch.nn.ReLU(),
      torch.nn.Linear(120, 84),
      torch.nn.ReLU(),
      torch.nn.Linear(84, len(classes))
    )

  def forward(self, x):
    return self.layers(x)


In [ ]:
model = ModelNN_v2()
model = model.to(device)
sum(p.numel() for p in model.parameters())

In [ ]:
transforms = tfs.Compose(
    [
        tfs.ToImage(),
        tfs.ToDtype(torch.float32, scale=True)
    ]
)

train_size = 0.8
batch_size = 100
album = False

In [ ]:
train_data, train_data_val, test_data = split_data(transforms, train_size=train_size, 
                                                   batch_size=batch_size, album=album)

In [ ]:
optimizer = optim.Adam(params=model.parameters(), lr=0.001)
loss_func = nn.CrossEntropyLoss()
epochs = 30

In [ ]:
model, res_lst, res_lst_val = train_model(model, device, train_data, train_data_val, 
                                            optimizer, loss_func, epochs)

In [ ]:
print(accuracy_model(model, device, test_data))
show_loss(res_lst, res_lst_val)

#### Модель с большим кол-вом сверточных слоев (в сравнении)

In [ ]:
model = ModelNN_cnn5()
model = model.to(device)
sum(p.numel() for p in model.parameters())

In [ ]:
transforms = tfs.Compose(
    [
        tfs.ToImage(),
        tfs.ToDtype(torch.float32, scale=True)
    ]
)

train_size = 0.8
batch_size = 100
album = False

In [ ]:
train_data, train_data_val, test_data = split_data(transforms, train_size=train_size, 
                                                   batch_size=batch_size, album=album)

In [ ]:
optimizer = optim.Adam(params=model.parameters(), lr=0.001)
loss_func = nn.CrossEntropyLoss()
epochs = 30

In [ ]:
model, res_lst, res_lst_val = train_model(model, device, train_data, train_data_val, 
                                            optimizer, loss_func, epochs)

In [ ]:
print(accuracy_model(model, device, test_data))
show_loss(res_lst, res_lst_val)

## Предотвращение переобучения

### Пример переобучения

In [ ]:
class ModelNN_mini(nn.Module):
  def __init__(self):
    super().__init__()
    self.layers = nn.Sequential(
      torch.nn.Conv2d(in_channels=3, out_channels=6, kernel_size=5, padding=2),  
      torch.nn.ReLU(),
      torch.nn.AvgPool2d(kernel_size=3, stride=3),  

      torch.nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5, padding=2), 
      torch.nn.ReLU(),
      torch.nn.AvgPool2d(kernel_size=3, stride=3),  

      nn.Flatten(),

      torch.nn.Linear(22 * 22 * 16, 256),
      torch.nn.ReLU(),

      torch.nn.Linear(256, 128),
      torch.nn.ReLU(),
      torch.nn.Linear(128, 90),
      torch.nn.ReLU(),

      torch.nn.Linear(90, 84),
      torch.nn.ReLU(),

      torch.nn.Linear(84, 50),
      torch.nn.ReLU(),

      torch.nn.Linear(50, len(classes))
    )

  def forward(self, x):
    return self.layers(x)


In [ ]:
model = ModelNN_mini()
model = model.to(device)
sum(p.numel() for p in model.parameters())

In [ ]:
optimizer = optim.Adam(params=model.parameters(), lr=0.001)
loss_func = nn.CrossEntropyLoss()
epochs = 30

In [ ]:
transforms = tfs.Compose(
    [
        tfs.ToImage(),
        tfs.ToDtype(torch.float32, scale=True)
    ]
)

train_size = 0.8
batch_size = 100
album = False

In [ ]:
train_data, train_data_val, test_data = split_data(transforms, train_size=train_size, 
                                                   batch_size=batch_size, album=album)

In [ ]:
model, res_lst, res_lst_val = train_model(model, device, train_data, train_data_val, 
                                            optimizer, loss_func, epochs)

In [ ]:
print(accuracy_model(model, device, test_data))
show_loss(res_lst, res_lst_val)

### Dropout

#### nn.Dropout2d(0.3)

In [ ]:
class ModelNN_mini_drop3(nn.Module):
  def __init__(self):
    super().__init__()
    self.layers = nn.Sequential(
      torch.nn.Conv2d(in_channels=3, out_channels=6, kernel_size=5, padding=2),  
      torch.nn.ReLU(),
      torch.nn.AvgPool2d(kernel_size=3, stride=3),  
      nn.Dropout2d(0.3),

      torch.nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5, padding=2), 
      torch.nn.ReLU(),
      torch.nn.AvgPool2d(kernel_size=3, stride=3),  
      nn.Dropout2d(0.3),

      nn.Flatten(),

      torch.nn.Linear(22 * 22 * 16, 256),
      torch.nn.ReLU(),
      nn.Dropout1d(0.3),

      torch.nn.Linear(256, 128),
      torch.nn.ReLU(),
      nn.Dropout1d(0.3),
      torch.nn.Linear(128, 90),
      torch.nn.ReLU(),
      nn.Dropout1d(0.3),

      torch.nn.Linear(90, 84),
      torch.nn.ReLU(),
      nn.Dropout1d(0.3),

      torch.nn.Linear(84, 50),
      torch.nn.ReLU(),
      nn.Dropout1d(0.3),

      torch.nn.Linear(50, len(classes))
    )

  def forward(self, x):
    return self.layers(x)


In [ ]:
model = ModelNN_mini_drop3()
model = model.to(device)
sum(p.numel() for p in model.parameters())

In [ ]:
optimizer = optim.Adam(params=model.parameters(), lr=0.001)
loss_func = nn.CrossEntropyLoss()
epochs = 30

In [ ]:
model, res_lst, res_lst_val = train_model(model, device, train_data, train_data_val, 
                                            optimizer, loss_func, epochs)

In [ ]:
print(accuracy_model(model, device, test_data))
show_loss(res_lst, res_lst_val)

#### nn.Dropout2d(0.15)

In [ ]:
class ModelNN_mini_drop15(nn.Module):
  def __init__(self):
    super().__init__()
    self.layers = nn.Sequential(
      torch.nn.Conv2d(in_channels=3, out_channels=6, kernel_size=5, padding=2),  
      torch.nn.ReLU(),
      torch.nn.AvgPool2d(kernel_size=3, stride=3),  
      nn.Dropout2d(0.15),

      torch.nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5, padding=2), 
      torch.nn.ReLU(),
      torch.nn.AvgPool2d(kernel_size=3, stride=3),  
      nn.Dropout2d(0.15),

      nn.Flatten(),

      torch.nn.Linear(22 * 22 * 16, 256),
      torch.nn.ReLU(),
      nn.Dropout1d(0.15),

      torch.nn.Linear(256, 128),
      torch.nn.ReLU(),
      nn.Dropout1d(0.15),
      torch.nn.Linear(128, 90),
      torch.nn.ReLU(),
      nn.Dropout1d(0.15),

      torch.nn.Linear(90, 84),
      torch.nn.ReLU(),
      nn.Dropout1d(0.15),

      torch.nn.Linear(84, 50),
      torch.nn.ReLU(),
      nn.Dropout1d(0.15),

      torch.nn.Linear(50, len(classes))
    )

  def forward(self, x):
    return self.layers(x)


In [ ]:
model = ModelNN_mini_drop15()
model = model.to(device)
sum(p.numel() for p in model.parameters())

In [ ]:
optimizer = optim.Adam(params=model.parameters(), lr=0.001)
loss_func = nn.CrossEntropyLoss()
epochs = 30

In [ ]:
model, res_lst, res_lst_val = train_model(model, device, train_data, train_data_val, 
                                            optimizer, loss_func, epochs)

In [ ]:
print(accuracy_model(model, device, test_data))
show_loss(res_lst, res_lst_val)

### BatchNorm

In [ ]:
class ModelNN_mini_bn(nn.Module):
  def __init__(self):
    super().__init__()
    self.layers = nn.Sequential(
      torch.nn.Conv2d(in_channels=3, out_channels=6, kernel_size=5, padding=2),  
      nn.BatchNorm2d(6),
      torch.nn.ReLU(),
      torch.nn.AvgPool2d(kernel_size=3, stride=3),  

      torch.nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5, padding=2), 
      nn.BatchNorm2d(16),
      torch.nn.ReLU(),
      torch.nn.AvgPool2d(kernel_size=3, stride=3),  

      nn.Flatten(),

      torch.nn.Linear(22 * 22 * 16, 256),
      nn.BatchNorm1d(256),
      torch.nn.ReLU(),

      torch.nn.Linear(256, 128),
      nn.BatchNorm1d(128),
      torch.nn.ReLU(),
      torch.nn.Linear(128, 90),
      nn.BatchNorm1d(90),
      torch.nn.ReLU(),

      torch.nn.Linear(90, 84),
      nn.BatchNorm1d(84),
      torch.nn.ReLU(),

      torch.nn.Linear(84, 50),
      nn.BatchNorm1d(50),
      torch.nn.ReLU(),

      torch.nn.Linear(50, len(classes))
    )

  def forward(self, x):
    return self.layers(x)


In [ ]:
model = ModelNN_mini_bn()
model = model.to(device)
sum(p.numel() for p in model.parameters())

In [ ]:
optimizer = optim.Adam(params=model.parameters(), lr=0.001)
loss_func = nn.CrossEntropyLoss()
epochs = 30

In [ ]:
model, res_lst, res_lst_val = train_model(model, device, train_data, train_data_val, 
                                            optimizer, loss_func, epochs)

In [ ]:
print(accuracy_model(model, device, test_data))
show_loss(res_lst, res_lst_val)

### Callbacks

#### Начальные функции и классы

In [ ]:
import torch


class ModelCheckpoint:
    def __init__(self, filepath, save_best_only=True, save_step=1, metric='accuracy'):
        self.filepath = filepath
        self.save_best_only = save_best_only
        self.save_step = save_step
        self.metric=metric

        if self.metric == 'accuracy':
            self.best = -float('inf')
            self.compare = lambda a, b: a > b
        elif self.metric == 'loss':
            self.best = float('inf')
            self.compare = lambda a, b: a < b

    def step(self, model, cur_metric_value, epoch):
        save_flag = False

        # Сохраняем по расписанию
        if self.save_step > 0 and (epoch + 1) % self.save_step == 0:
            # Сохраняем при улучшении метрики
            if self.save_best_only:
                if self.compare(cur_metric_value, self.best):
                    self.best = cur_metric_value
                    save_flag = True
            else:
                save_flag = True

        if save_flag:
            torch.save(model.state_dict(), 
                       self.filepath.format(epoch=epoch + 1, metric=self.metric, metric_val=cur_metric_value))
            print("ModelCheckpoint: model saved")

In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, save_best=True, filepath=None, metric='accuracy'):
        self.patience = patience
        self.save_best = save_best
        self.filepath = filepath
        self.metric=metric

        self.counter = 0
        self.best_val = None
        self.best_model = None
        self.early_stop = False

        if self.metric == 'accuracy':
            self.best_val = -float('inf')
            self.compare = lambda a, b: a > b
        elif self.metric == 'loss':
            self.best_val = float('inf')
            self.compare = lambda a, b: a < b

    def step(self, model, cur_metric_value, epoch):
        if self.compare(cur_metric_value, self.best_val):
            self.best_val = cur_metric_value
            self.best_model = copy.deepcopy(model.state_dict())
            self.counter = 0
        else:
            self.counter += 1
            print(f"EarlyStopping: no improvement for {self.counter} epochs")

        if self.counter >= self.patience:
            self.early_stop = True
            if self.save_best:
                torch.save(self.best_model, 
                        self.filepath.format(metric=self.metric, metric_val=self.best_val))
            else:
                torch.save(model.state_dict(), 
                        self.filepath.format(metric=self.metric, metric_val=cur_metric_value))
            print(f"EarlyStopping: stopping training at epoch {epoch + 1}")

In [ ]:
def update_scheduler(lr_scheduler, val_res):
    if lr_scheduler is not None:
        if isinstance(lr_scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
            lr_scheduler.step(val_res)
        else:
            lr_scheduler.step()

def do_callbacks(model, callbacks, epoch, val_res):
    for cb in callbacks:
        # фиксируем чекпоинты
        if isinstance(cb, ModelCheckpoint):
            cb.step(model, val_res, epoch)

        # проверяем на улучшение
        elif isinstance(cb, EarlyStopping):
            cb.step(model, val_res, epoch)

            if cb.early_stop:
                if cb.save_best == True:
                    model.load_state_dict(cb.best_model)
                return True
    return False

def train_model_with_callbacks(
    model, device, 
    train_data, train_size, train_data_val,
    optimizer, loss_func, epochs,
    metric='accuracy',
    callbacks=[], lr_scheduler=None
):
    model.to(device)
    res_lst_val = []
    res_lst = []

    for epoch in range(epochs):
        model.train()
        loss_mean = 0
        lm_count = 0
        train_tqdm = tqdm(train_data, leave=False)

        for x_train, y_train in train_tqdm:
            x_train = x_train.to(device)
            y_train = y_train.to(device)

            predict = model(x_train)
            loss = loss_func(predict, y_train)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            lm_count += 1
            loss_mean = loss.item() / lm_count + loss_mean * (1 - 1 / lm_count)
            train_tqdm.set_description(f"Epoch [{epoch+1}/{epochs}], loss: {loss_mean:.4f}")

        # метрики на обучении и валидации
        if metric == 'accuracy':
            train_res = accuracy_model(model, device, train_data)
            val_res = accuracy_model(model, device, train_data_val)
        elif metric == 'loss':
            train_res = loss_mean
            val_res = loss_model(model, loss_func, train_data_val)

        res_lst.append(train_res)
        res_lst_val.append(val_res)

        print(f'Epoch [{epoch+1}/{epochs}] | {metric}_train={train_res:.3f}, {metric}_val={val_res:.3f}')

        # callbacks
        early_stop = do_callbacks(model, callbacks, epoch, val_res)
        if early_stop:
            return model, res_lst, res_lst_val
        
        # scheduler
        update_scheduler(lr_scheduler, val_res)

    return model, res_lst, res_lst_val


#### Пример реализации

In [ ]:
model = ModelNN_mini_bn()
model = model.to(device)
sum(p.numel() for p in model.parameters())

In [ ]:
optimizer = optim.Adam(params=model.parameters(), lr=0.001)
loss_func = nn.CrossEntropyLoss()
epochs = 30

In [ ]:
transforms = tfs.Compose(
    [
        tfs.ToImage(),
        tfs.ToDtype(torch.float32, scale=True)
    ]
)

train_size = 0.8
batch_size = 100
album = False

In [ ]:
train_data, train_data_val, test_data = split_data(transforms, train_size=train_size, 
                                                   batch_size=batch_size, album=album)

In [ ]:
models_path = 'models'
chck_p_path = os.path.join(models_path, 'model_epoch-{epoch}_{metric}-{metric_val:.4f}.pt')
erly_st_path = os.path.join(models_path, 'best_model_{metric}-{metric_val:.4f}.pt')

metric = 'loss'

checkpoint_cb = ModelCheckpoint(
    filepath=chck_p_path, 
    save_best_only=True, 
    save_step=3,
    metric=metric
)

earlystop_cb = EarlyStopping(
    filepath=erly_st_path, 
    patience=1, 
    save_best=True,
    metric=metric
)

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

model, res_lst, res_lst_val = train_model_with_callbacks(
    model, device, train_data, train_size, train_data_val,
    optimizer, loss_func, epochs=50,
    metric=metric,
    callbacks=[earlystop_cb],
    lr_scheduler=scheduler
)


In [ ]:
print(accuracy_model(model, device, test_data))
show_loss(res_lst, res_lst_val)

#### patience=1

In [ ]:
model = ModelNN_mini_bn()
model = model.to(device)
sum(p.numel() for p in model.parameters())

In [ ]:
optimizer = optim.Adam(params=model.parameters(), lr=0.001)
loss_func = nn.CrossEntropyLoss()
epochs = 30

In [ ]:
transforms = tfs.Compose(
    [
        tfs.ToImage(),
        tfs.ToDtype(torch.float32, scale=True)
    ]
)

train_size = 0.8
batch_size = 100
album = False

In [ ]:
train_data, train_data_val, test_data = split_data(transforms, train_size=train_size, 
                                                   batch_size=batch_size, album=album)

In [ ]:
models_path = 'models'
chck_p_path = os.path.join(models_path, 'model_epoch-{epoch}_{metric}-{metric_val:.4f}.pt')
erly_st_path = os.path.join(models_path, 'best_model_{metric}-{metric_val:.4f}.pt')

metric = 'accuracy'

checkpoint_cb = ModelCheckpoint(
    filepath=chck_p_path, 
    save_best_only=True, 
    save_step=3,
    metric=metric
)

earlystop_cb = EarlyStopping(
    filepath=erly_st_path, 
    patience=1, 
    save_best=True,
    metric=metric
)

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

model, res_lst, res_lst_val = train_model_with_callbacks(
    model, device, train_data, train_size, train_data_val,
    optimizer, loss_func, epochs=50,
    metric=metric,
    callbacks=[earlystop_cb],
    lr_scheduler=scheduler
)


In [ ]:
print(accuracy_model(model, device, test_data))
show_loss(res_lst, res_lst_val)

#### patience=5

In [ ]:
model = ModelNN_mini_bn()
model = model.to(device)
sum(p.numel() for p in model.parameters())

In [ ]:
optimizer = optim.Adam(params=model.parameters(), lr=0.001)
loss_func = nn.CrossEntropyLoss()
epochs = 30

In [ ]:
transforms = tfs.Compose(
    [
        tfs.ToImage(),
        tfs.ToDtype(torch.float32, scale=True)
    ]
)

train_size = 0.8
batch_size = 100
album = False

In [ ]:
train_data, train_data_val, test_data = split_data(transforms, train_size=train_size, 
                                                   batch_size=batch_size, album=album)

In [ ]:
models_path = 'models'
chck_p_path = os.path.join(models_path, 'model_epoch-{epoch}_{metric}-{metric_val:.4f}.pt')
erly_st_path = os.path.join(models_path, 'best_model_{metric}-{metric_val:.4f}.pt')

metric = 'accuracy'

checkpoint_cb = ModelCheckpoint(
    filepath=chck_p_path, 
    save_best_only=True, 
    save_step=3,
    metric=metric
)

earlystop_cb = EarlyStopping(
    filepath=erly_st_path, 
    patience=5, 
    save_best=True,
    metric=metric
)

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

model, res_lst, res_lst_val = train_model_with_callbacks(
    model, device, train_data, train_size, train_data_val,
    optimizer, loss_func, epochs=50,
    metric=metric,
    callbacks=[earlystop_cb],
    lr_scheduler=scheduler
)


In [ ]:
print(accuracy_model(model, device, test_data))
show_loss(res_lst, res_lst_val)

#### patience=10

In [ ]:
model = ModelNN_mini_bn()
model = model.to(device)
sum(p.numel() for p in model.parameters())

In [ ]:
optimizer = optim.Adam(params=model.parameters(), lr=0.001)
loss_func = nn.CrossEntropyLoss()
epochs = 30

In [ ]:
transforms = tfs.Compose(
    [
        tfs.ToImage(),
        tfs.ToDtype(torch.float32, scale=True)
    ]
)

train_size = 0.8
batch_size = 100
album = False

In [ ]:
train_data, train_data_val, test_data = split_data(transforms, train_size=train_size, 
                                                   batch_size=batch_size, album=album)

In [ ]:
models_path = 'models'
chck_p_path = os.path.join(models_path, 'model_epoch-{epoch}_{metric}-{metric_val:.4f}.pt')
erly_st_path = os.path.join(models_path, 'best_model_{metric}-{metric_val:.4f}.pt')

metric = 'accuracy'

checkpoint_cb = ModelCheckpoint(
    filepath=chck_p_path, 
    save_best_only=True, 
    save_step=3,
    metric=metric
)

earlystop_cb = EarlyStopping(
    filepath=erly_st_path, 
    patience=10, 
    save_best=True,
    metric=metric
)

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

model, res_lst, res_lst_val = train_model_with_callbacks(
    model, device, train_data, train_size, train_data_val,
    optimizer, loss_func, epochs=50,
    metric=metric,
    callbacks=[earlystop_cb],
    lr_scheduler=scheduler
)


In [ ]:
print(accuracy_model(model, device, test_data))
show_loss(res_lst, res_lst_val)

### Аугментация

#### Базовые функции

In [ ]:
class AlbumentationsTransform:
    def __init__(self, albumentations_transform):
        self.albumentations_transform = albumentations_transform

    def __call__(self, img):
        img = np.array(img)
        augmented = self.albumentations_transform(image=img)
        return augmented['image']

In [ ]:
class ModelNN_mini_bn(nn.Module):
  def __init__(self):
    super().__init__()
    self.layers = nn.Sequential(
      torch.nn.Conv2d(in_channels=3, out_channels=6, kernel_size=5, padding=2),  
      nn.BatchNorm2d(6),
      torch.nn.ReLU(),
      torch.nn.AvgPool2d(kernel_size=3, stride=3),  

      torch.nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5, padding=2), 
      nn.BatchNorm2d(16),
      torch.nn.ReLU(),
      torch.nn.AvgPool2d(kernel_size=3, stride=3),  

      nn.Flatten(),

      torch.nn.Linear(24 * 24 * 16, 256),
      nn.BatchNorm1d(256),
      torch.nn.ReLU(),

      torch.nn.Linear(256, 128),
      nn.BatchNorm1d(128),
      torch.nn.ReLU(),
      torch.nn.Linear(128, 90),
      nn.BatchNorm1d(90),
      torch.nn.ReLU(),

      torch.nn.Linear(90, 84),
      nn.BatchNorm1d(84),
      torch.nn.ReLU(),

      torch.nn.Linear(84, 50),
      nn.BatchNorm1d(50),
      torch.nn.ReLU(),

      torch.nn.Linear(50, len(classes))
    )

  def forward(self, x):
    return self.layers(x)


#### Положение камеры

In [ ]:
model = ModelNN_mini_bn()
model = model.to(device)
sum(p.numel() for p in model.parameters())

In [ ]:
optimizer = optim.Adam(params=model.parameters(), lr=0.001)
loss_func = nn.CrossEntropyLoss()
epochs = 30

In [ ]:
transforms = A.Compose(
    [
        A.Affine(
            translate_percent={"x": (-0.15, 0.15), "y": (-0.15, 0.15)},
            scale=(0.85, 1.15),
            rotate=(-15, 15),
            p=0.5),
        A.Resize(224, 224),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ]
)

train_size = 0.8
batch_size = 100
album = True

In [ ]:
train_data, train_data_val, test_data = split_data(transforms, train_size=train_size, 
                                                   batch_size=batch_size, album=album)

In [ ]:
data_iter = iter(train_data)
images, labels = next(data_iter)

img = images[0]  # тензор [C, H, W]

img = img.permute(1, 2, 0).cpu().numpy()  # [H, W, C]
img = img * 0.5 + 0.5
img = np.clip(img, 0, 1)

plt.imshow(img.squeeze(), cmap='gray')
plt.title(f'Label: {labels[0].item()}')
plt.axis('off')
plt.show()


In [ ]:
models_path = 'models'
chck_p_path = os.path.join(models_path, 'model_epoch-{epoch}_{metric}-{metric_val:.4f}.pt')
erly_st_path = os.path.join(models_path, 'best_model_{metric}-{metric_val:.4f}.pt')

metric = 'accuracy'

checkpoint_cb = ModelCheckpoint(
    filepath=chck_p_path, 
    save_best_only=True, 
    save_step=3,
    metric=metric
)

earlystop_cb = EarlyStopping(
    filepath=erly_st_path, 
    patience=5, 
    save_best=True,
    metric=metric
)

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

model, res_lst, res_lst_val = train_model_with_callbacks(
    model, device, train_data, train_size, train_data_val,
    optimizer, loss_func, epochs=50,
    metric=metric,
    callbacks=[earlystop_cb],
    lr_scheduler=scheduler
)


In [ ]:
print(accuracy_model(model, device, test_data))
show_loss(res_lst, res_lst_val)

#### Яркость изображения

In [ ]:
model = ModelNN_mini_bn()
model = model.to(device)
sum(p.numel() for p in model.parameters())

In [ ]:
optimizer = optim.Adam(params=model.parameters(), lr=0.001)
loss_func = nn.CrossEntropyLoss()
epochs = 30

In [ ]:
transforms = A.Compose(
    [
        A.RandomBrightnessContrast(p=0.2),
        A.Resize(224, 224),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ]
)

train_size = 0.8
batch_size = 100
album = True

In [ ]:
train_data, train_data_val, test_data = split_data(transforms, train_size=train_size, 
                                                   batch_size=batch_size, album=album)

In [ ]:
data_iter = iter(train_data)
images, labels = next(data_iter)

img = images[0]  # тензор [C, H, W]

img = img.permute(1, 2, 0).cpu().numpy()  # [H, W, C]
img = img * 0.5 + 0.5
img = np.clip(img, 0, 1)

plt.imshow(img.squeeze(), cmap='gray')
plt.title(f'Label: {labels[0].item()}')
plt.axis('off')
plt.show()


In [ ]:
models_path = 'models'
chck_p_path = os.path.join(models_path, 'model_epoch-{epoch}_{metric}-{metric_val:.4f}.pt')
erly_st_path = os.path.join(models_path, 'best_model_{metric}-{metric_val:.4f}.pt')

metric = 'accuracy'

checkpoint_cb = ModelCheckpoint(
    filepath=chck_p_path, 
    save_best_only=True, 
    save_step=3,
    metric=metric
)

earlystop_cb = EarlyStopping(
    filepath=erly_st_path, 
    patience=5, 
    save_best=True,
    metric=metric
)

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

model, res_lst, res_lst_val = train_model_with_callbacks(
    model, device, train_data, train_size, train_data_val,
    optimizer, loss_func, epochs=50,
    metric=metric,
    callbacks=[earlystop_cb],
    lr_scheduler=scheduler
)


In [ ]:
print(accuracy_model(model, device, test_data))
show_loss(res_lst, res_lst_val)

#### Искажения камеры

In [ ]:
model = ModelNN_mini_bn()
model = model.to(device)
sum(p.numel() for p in model.parameters())

In [ ]:
optimizer = optim.Adam(params=model.parameters(), lr=0.001)
loss_func = nn.CrossEntropyLoss()
epochs = 30

In [ ]:
transforms = A.Compose(
    [
        A.OneOf([
            A.MotionBlur(p=0.2),
            A.OpticalDistortion(p=0.2),
            A.GaussNoise(p=0.2)
        ], p=1),
        A.Resize(224, 224),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ]
)

train_size = 0.8
batch_size = 100
album = True

In [ ]:
train_data, train_data_val, test_data = split_data(transforms, train_size=train_size, 
                                                   batch_size=batch_size, album=album)

In [ ]:
data_iter = iter(train_data)
images, labels = next(data_iter)

img = images[0]  # тензор [C, H, W]

img = img.permute(1, 2, 0).cpu().numpy()  # [H, W, C]
img = img * 0.5 + 0.5
img = np.clip(img, 0, 1)

plt.imshow(img.squeeze(), cmap='gray')
plt.title(f'Label: {labels[0].item()}')
plt.axis('off')
plt.show()


In [ ]:
models_path = 'models'
chck_p_path = os.path.join(models_path, 'model_epoch-{epoch}_{metric}-{metric_val:.4f}.pt')
erly_st_path = os.path.join(models_path, 'best_model_{metric}-{metric_val:.4f}.pt')

metric = 'accuracy'

checkpoint_cb = ModelCheckpoint(
    filepath=chck_p_path, 
    save_best_only=True, 
    save_step=3,
    metric=metric
)

earlystop_cb = EarlyStopping(
    filepath=erly_st_path, 
    patience=5, 
    save_best=True,
    metric=metric
)

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

model, res_lst, res_lst_val = train_model_with_callbacks(
    model, device, train_data, train_size, train_data_val,
    optimizer, loss_func, epochs=50,
    metric=metric,
    callbacks=[earlystop_cb],
    lr_scheduler=scheduler
)


In [ ]:
print(accuracy_model(model, device, test_data, test_count))
show_loss(res_lst, res_lst_val)

#### Все вместе

In [ ]:
model = ModelNN_mini_bn()
model = model.to(device)
sum(p.numel() for p in model.parameters())

In [ ]:
optimizer = optim.Adam(params=model.parameters(), lr=0.001)
loss_func = nn.CrossEntropyLoss()
epochs = 30

In [ ]:
transforms = A.Compose(
    [
        A.Affine(
            translate_percent={"x": (-0.15, 0.15), "y": (-0.15, 0.15)},
            scale=(0.85, 1.15),
            rotate=(-15, 15),
            p=0.5),
        A.RandomBrightnessContrast(p=0.2),
        A.OneOf([
            A.MotionBlur(p=0.2),
            A.OpticalDistortion(p=0.2),
            A.GaussNoise(p=0.2)
        ], p=1),
        A.Resize(224, 224),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ]
)

train_size = 0.8
batch_size = 100
album = True

In [ ]:
train_data, train_data_val, test_data = split_data(transforms, train_size=train_size, 
                                                   batch_size=batch_size, album=album)

In [ ]:
data_iter = iter(train_data)
images, labels = next(data_iter)

img = images[0]  # тензор [C, H, W]

img = img.permute(1, 2, 0).cpu().numpy()  # [H, W, C]
img = img * 0.5 + 0.5
img = np.clip(img, 0, 1)

plt.imshow(img.squeeze(), cmap='gray')
plt.title(f'Label: {labels[0].item()}')
plt.axis('off')
plt.show()


In [ ]:
models_path = 'models'
chck_p_path = os.path.join(models_path, 'model_epoch-{epoch}_{metric}-{metric_val:.4f}.pt')
erly_st_path = os.path.join(models_path, 'best_model_{metric}-{metric_val:.4f}.pt')

metric = 'accuracy'

checkpoint_cb = ModelCheckpoint(
    filepath=chck_p_path, 
    save_best_only=True, 
    save_step=3,
    metric=metric
)

earlystop_cb = EarlyStopping(
    filepath=erly_st_path, 
    patience=5, 
    save_best=True,
    metric=metric
)

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

model, res_lst, res_lst_val = train_model_with_callbacks(
    model, device, train_data, train_size, train_data_val,
    optimizer, loss_func, epochs=50,
    metric=metric,
    callbacks=[earlystop_cb],
    lr_scheduler=scheduler
)


In [ ]:
print(accuracy_model(model, device, test_data))
show_loss(res_lst, res_lst_val)

## Использование предобученных моделей нейронных сетей

### Использование моделей нейронной сети из библиотеки torchvision

#### ShuffleNet

In [ ]:
model = torchvision.models.shufflenet_v2_x1_5().to(device)

In [ ]:
optimizer = optim.Adam(params=model.parameters(), lr=0.001)
loss_func = nn.CrossEntropyLoss()

In [ ]:
transforms = A.Compose(
    [
        A.Affine(
            translate_percent={"x": (-0.15, 0.15), "y": (-0.15, 0.15)},
            scale=(0.85, 1.15),
            rotate=(-15, 15),
            p=0.5),
        A.RandomBrightnessContrast(p=0.2),
        A.OneOf([
            A.MotionBlur(p=0.2),
            A.OpticalDistortion(p=0.2),
            A.GaussNoise(p=0.2)
        ], p=1),
        A.Resize(224, 224),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ]
)

train_size = 0.8
batch_size = 100
album = True

In [ ]:
train_data, train_data_val, test_data = split_data(transforms, train_size=train_size, 
                                                   batch_size=batch_size, album=album)

In [ ]:
models_path = 'models'
chck_p_path = os.path.join(models_path, 'model_epoch-{epoch}_{metric}-{metric_val:.4f}.pt')
erly_st_path = os.path.join(models_path, 'best_model_{metric}-{metric_val:.4f}.pt')

metric = 'accuracy'

checkpoint_cb = ModelCheckpoint(
    filepath=chck_p_path, 
    save_best_only=True, 
    save_step=3,
    metric=metric
)

earlystop_cb = EarlyStopping(
    filepath=erly_st_path, 
    patience=5, 
    save_best=True,
    metric=metric
)

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

model, res_lst, res_lst_val = train_model_with_callbacks(
    model, device, train_data, train_size, train_data_val,
    optimizer, loss_func, epochs=50,
    metric=metric,
    callbacks=[earlystop_cb],
    lr_scheduler=scheduler
)


In [ ]:
print(accuracy_model(model, device, test_data))
show_loss(res_lst, res_lst_val)

#### RegNet

In [ ]:
model = torchvision.models.regnet_y_400mf().to(device)

In [ ]:
optimizer = optim.Adam(params=model.parameters(), lr=0.001)
loss_func = nn.CrossEntropyLoss()

In [ ]:
transforms = A.Compose(
    [
        A.Affine(
            translate_percent={"x": (-0.15, 0.15), "y": (-0.15, 0.15)},
            scale=(0.85, 1.15),
            rotate=(-15, 15),
            p=0.5),
        A.RandomBrightnessContrast(p=0.2),
        A.OneOf([
            A.MotionBlur(p=0.2),
            A.OpticalDistortion(p=0.2),
            A.GaussNoise(p=0.2)
        ], p=1),
        A.Resize(224, 224),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ]
)

train_size = 0.8
batch_size = 100
album = True

In [ ]:
train_data, train_data_val, test_data = split_data(transforms, train_size=train_size, 
                                                   batch_size=batch_size, album=album)

In [ ]:
models_path = 'models'
chck_p_path = os.path.join(models_path, 'model_epoch-{epoch}_{metric}-{metric_val:.4f}.pt')
erly_st_path = os.path.join(models_path, 'best_model_{metric}-{metric_val:.4f}.pt')

metric = 'accuracy'

checkpoint_cb = ModelCheckpoint(
    filepath=chck_p_path, 
    save_best_only=True, 
    save_step=3,
    metric=metric
)

earlystop_cb = EarlyStopping(
    filepath=erly_st_path, 
    patience=5, 
    save_best=True,
    metric=metric
)

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

model, res_lst, res_lst_val = train_model_with_callbacks(
    model, device, train_data, train_size, train_data_val,
    optimizer, loss_func, epochs=50,
    metric=metric,
    callbacks=[earlystop_cb],
    lr_scheduler=scheduler
)


In [ ]:
print(accuracy_model(model, device, test_data))
show_loss(res_lst, res_lst_val)

### Перенос обучения

#### ShuffleNet

In [ ]:
weights = torchvision.models.ShuffleNet_V2_X1_5_Weights.DEFAULT
model = torchvision.models.shufflenet_v2_x1_5(weights=weights).to(device)

In [ ]:
for param in model.conv1.parameters():
  param.requires_grad = False
for param in model.stage2.parameters():
  param.requires_grad = False
for param in model.stage3.parameters():
  param.requires_grad = False
for param in model.stage4.parameters():
  param.requires_grad = False
for param in model.conv5.parameters():
  param.requires_grad = False

In [ ]:
num_features = model.fc.in_features 
model.fc = torch.nn.Linear(num_features, len(classes)).to(device)

In [ ]:
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)
loss_func = nn.CrossEntropyLoss()

In [ ]:
transforms = A.Compose(
    [
        A.Affine(
            translate_percent={"x": (-0.15, 0.15), "y": (-0.15, 0.15)},
            scale=(0.85, 1.15),
            rotate=(-15, 15),
            p=0.5),
        A.RandomBrightnessContrast(p=0.2),
        A.OneOf([
            A.MotionBlur(p=0.2),
            A.OpticalDistortion(p=0.2),
            A.GaussNoise(p=0.2)
        ], p=1),
        A.Resize(224, 224),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ]
)

train_size = 0.8
batch_size = 100
album = True

In [ ]:
train_data, train_data_val, test_data = split_data(transforms, train_size=train_size, 
                                                   batch_size=batch_size, album=album)

In [ ]:
models_path = 'models'
chck_p_path = os.path.join(models_path, 'model_epoch-{epoch}_{metric}-{metric_val:.4f}.pt')
erly_st_path = os.path.join(models_path, 'best_model_{metric}-{metric_val:.4f}.pt')

metric = 'accuracy'

checkpoint_cb = ModelCheckpoint(
    filepath=chck_p_path, 
    save_best_only=True, 
    save_step=3,
    metric=metric
)

earlystop_cb = EarlyStopping(
    filepath=erly_st_path, 
    patience=5, 
    save_best=True,
    metric=metric
)

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

model, res_lst, res_lst_val = train_model_with_callbacks(
    model, device, train_data, train_size, train_data_val,
    optimizer, loss_func, epochs=50,
    metric=metric,
    callbacks=[earlystop_cb],
    lr_scheduler=scheduler
)


In [ ]:
print(accuracy_model(model, device, test_data))
show_loss(res_lst, res_lst_val)

#### RegNet

In [ ]:
weights = torchvision.models.RegNet_Y_400MF_Weights.DEFAULT
model = torchvision.models.regnet_y_400mf(weights=weights).to(device)

In [ ]:
for param in model.stem.parameters():
  param.requires_grad = False
for param in model.trunk_output.parameters():
  param.requires_grad = False

In [ ]:
num_features = model.fc.in_features
model.fc = torch.nn.Linear(num_features, len(classes)).to(device)

In [ ]:
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)
loss_func = nn.CrossEntropyLoss()

In [ ]:
transforms = A.Compose(
    [
        A.Affine(
            translate_percent={"x": (-0.15, 0.15), "y": (-0.15, 0.15)},
            scale=(0.85, 1.15),
            rotate=(-15, 15),
            p=0.5),
        A.RandomBrightnessContrast(p=0.2),
        A.OneOf([
            A.MotionBlur(p=0.2),
            A.OpticalDistortion(p=0.2),
            A.GaussNoise(p=0.2)
        ], p=1),
        A.Resize(224, 224),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ]
)

train_size = 0.8
batch_size = 100
album = True

In [ ]:
train_data, train_data_val, test_data = split_data(transforms, train_size=train_size, 
                                                   batch_size=batch_size, album=album)

In [ ]:
models_path = 'models'
chck_p_path = os.path.join(models_path, 'model_epoch-{epoch}_{metric}-{metric_val:.4f}.pt')
erly_st_path = os.path.join(models_path, 'best_model_{metric}-{metric_val:.4f}.pt')

metric = 'accuracy'

checkpoint_cb = ModelCheckpoint(
    filepath=chck_p_path, 
    save_best_only=True, 
    save_step=3,
    metric=metric
)

earlystop_cb = EarlyStopping(
    filepath=erly_st_path, 
    patience=5, 
    save_best=True,
    metric=metric
)

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

model, res_lst, res_lst_val = train_model_with_callbacks(
    model, device, train_data, train_size, train_data_val,
    optimizer, loss_func, epochs=50,
    metric=metric,
    callbacks=[earlystop_cb],
    lr_scheduler=scheduler
)


In [ ]:
print(accuracy_model(model, device, test_data))
show_loss(res_lst, res_lst_val)

### Fine tuning

#### Базовые функции

In [ ]:
def get_submodule_names(module, max_depth=2, prefix=''):
    # вывод списка названий суб-блоков НН 
    if max_depth == 0:
        return []
    
    names = []
    for name, child in module.named_children():
        if (name == 'fc') or ('pool' in name):
            continue
        
        full_name = f"{prefix}{name}" if prefix == '' else f"{prefix}.{name}"
        
        # Проверяем, есть ли у ребенка trainable параметры
        children = list(child.named_children())
        if max_depth > 1 and children:
            # Есть дети — рекурсивно ищем в них
            names.extend(get_submodule_names(child, max_depth - 1, full_name))
        else:
            # Дети есть, но max_depth исчерпан, или детей нет — добавляем этот модуль
            names.append(full_name)

    return names

In [ ]:
def get_submodule_by_name(model, submodule_name):
    # вывод блока НН по его названию
    parts = submodule_name.split('.')
    module = model
    for part in parts:
        module = getattr(module, part, None)
    return module

# Функция разморозки блока
def unfreeze_block(model, block_name):
    block = get_submodule_by_name(model, block_name)
    if block is None:
        return False

    # Проверка, разморожен ли уже блок
    any_frozen = any(not param.requires_grad for param in block.parameters())
    if not any_frozen:
        return False

    for param in block.parameters():
        param.requires_grad = True
    return True  


# Функция размораживания следующих блоков по номеру эпохи
def gradual_unfreeze(epoch, freeze_order, unfreeze_step=5):
    idx = epoch // unfreeze_step - 1
    if 0 <= idx < len(freeze_order):
        # вывод при успешной разморозке
        if unfreeze_block(model, freeze_order[idx]):
            print(f'Unfroze block: {freeze_order[idx]} at epoch {epoch}')

In [ ]:
def train_model_with_callbacks_finetune(
    model, device, 
    train_data, train_data_val,
    loss_func, epochs,
    freeze_order, unfreeze_step=5,
    opt_lr=0.001, lr_sch_step=5, lr_sch_gamma=0.1,
    metric='accuracy',
    callbacks=[],
):
    model.to(device)
    res_lst_val = []
    res_lst = []

    for epoch in range(epochs):
        model.train()
        loss_mean = 0
        lm_count = 0
        train_tqdm = tqdm(train_data, leave=False)

        gradual_unfreeze(epoch, freeze_order, unfreeze_step)
        optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=opt_lr)
        lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=lr_sch_step, gamma=lr_sch_gamma)

        for x_train, y_train in train_tqdm:
            x_train = x_train.to(device)
            y_train = y_train.to(device)

            predict = model(x_train)
            loss = loss_func(predict, y_train)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            lm_count += 1
            loss_mean = loss.item() / lm_count + loss_mean * (1 - 1 / lm_count)
            train_tqdm.set_description(f"Epoch [{epoch+1}/{epochs}], loss: {loss_mean:.4f}")

        # метрики на обучении и валидации
        if metric == 'accuracy':
            train_res = accuracy_model(model, device, train_data)
            val_res = accuracy_model(model, device, train_data_val)
        elif metric == 'loss':
            train_res = loss_mean
            val_res = loss_model(model, loss_func, train_data_val)

        res_lst.append(train_res)
        res_lst_val.append(val_res)

        print(f'Epoch [{epoch+1}/{epochs}] | {metric}_train={train_res:.3f}, {metric}_val={val_res:.3f}')

        # callbacks
        early_stop = do_callbacks(model, callbacks, epoch, val_res)
        if early_stop:
            return model, res_lst, res_lst_val
        
        # scheduler
        update_scheduler(lr_scheduler, val_res)
    return model, res_lst, res_lst_val

#### ShuffleNet

In [ ]:
weights = torchvision.models.ShuffleNet_V2_X1_5_Weights.DEFAULT
model = torchvision.models.shufflenet_v2_x1_5(weights=weights).to(device)

In [ ]:
summary(model=model,
        input_size=(100, 3, 224, 224),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"]
)

In [ ]:
for param in model.parameters():
    param.requires_grad = False

# Разморозить fc слой
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, len(classes)).to(device)
for param in model.fc.parameters():
    param.requires_grad = True

In [ ]:
loss_func = nn.CrossEntropyLoss()
epochs = 50

freeze_order = get_submodule_names(model, max_depth=1)[::-1]
print(len(freeze_order), freeze_order)

In [ ]:
transforms = A.Compose(
    [
        A.Affine(
            translate_percent={"x": (-0.15, 0.15), "y": (-0.15, 0.15)},
            scale=(0.85, 1.15),
            rotate=(-15, 15),
            p=0.5),
        A.RandomBrightnessContrast(p=0.2),
        A.OneOf([
            A.MotionBlur(p=0.2),
            A.OpticalDistortion(p=0.2),
            A.GaussNoise(p=0.2)
        ], p=1),
        A.Resize(224, 224),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ]
)

train_size = 0.8
batch_size = 100
album = True

In [ ]:
train_data, train_data_val, test_data = split_data(transforms, train_size=train_size, 
                                                   batch_size=batch_size, album=album)

In [ ]:
models_path = 'models'
chck_p_path = os.path.join(models_path, 'model_epoch-{epoch}_{metric}-{metric_val:.4f}.pt')
erly_st_path = os.path.join(models_path, 'best_model_{metric}-{metric_val:.4f}.pt')

metric = 'accuracy'

checkpoint_cb = ModelCheckpoint(
    filepath=chck_p_path, 
    save_best_only=True, 
    save_step=3,
    metric=metric
)

earlystop_cb = EarlyStopping(
    filepath=erly_st_path, 
    patience=5, 
    save_best=True,
    metric=metric
)

model, res_lst, res_lst_val = train_model_with_callbacks_finetune(
    model, device, train_data, train_data_val,
    loss_func, epochs,
    freeze_order, unfreeze_step=5,
    opt_lr=0.001, lr_sch_step=2, lr_sch_gamma=0.1,
    metric=metric,
    callbacks=[earlystop_cb]
)


In [ ]:
print(accuracy_model(model, device, test_data))
show_loss(res_lst, res_lst_val)

#### RegNet

In [ ]:
weights = torchvision.models.RegNet_Y_400MF_Weights.DEFAULT
model = torchvision.models.regnet_y_400mf(weights=weights).to(device)

In [ ]:
summary(model=model,
        input_size=(100, 3, 224, 224),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"]
)

In [ ]:
for param in model.parameters():
    param.requires_grad = False

# Разморозить fc слой
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, len(classes)).to(device)
for param in model.fc.parameters():
    param.requires_grad = True

In [ ]:
summary(model=model,
        input_size=(100, 3, 224, 224),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"]
)

In [ ]:
loss_func = nn.CrossEntropyLoss()
epochs = 50

freeze_order = get_submodule_names(model, max_depth=3)[::-1]
print(freeze_order)

In [ ]:
transforms = A.Compose(
    [
        A.Affine(
            translate_percent={"x": (-0.15, 0.15), "y": (-0.15, 0.15)},
            scale=(0.85, 1.15),
            rotate=(-15, 15),
            p=0.5),
        A.RandomBrightnessContrast(p=0.2),
        A.OneOf([
            A.MotionBlur(p=0.2),
            A.OpticalDistortion(p=0.2),
            A.GaussNoise(p=0.2)
        ], p=1),
        A.Resize(224, 224),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ]
)

train_size = 0.8
batch_size = 100
album = True

In [ ]:
train_data, train_data_val, test_data = split_data(transforms, train_size=train_size, 
                                                   batch_size=batch_size, album=album)

In [ ]:
models_path = 'models'
chck_p_path = os.path.join(models_path, 'model_epoch-{epoch}_{metric}-{metric_val:.4f}.pt')
erly_st_path = os.path.join(models_path, 'best_model_{metric}-{metric_val:.4f}.pt')

metric = 'accuracy'

checkpoint_cb = ModelCheckpoint(
    filepath=chck_p_path, 
    save_best_only=True, 
    save_step=3,
    metric=metric
)

earlystop_cb = EarlyStopping(
    filepath=erly_st_path, 
    patience=5, 
    save_best=True,
    metric=metric
)

model, res_lst, res_lst_val = train_model_with_callbacks_finetune(
    model, device, train_data, train_data_val,
    loss_func, epochs,
    freeze_order, unfreeze_step=4,
    opt_lr=0.001, lr_sch_step=2, lr_sch_gamma=0.1,
    metric=metric,
    callbacks=[earlystop_cb]
)


In [ ]:
print(accuracy_model(model, device, test_data))
show_loss(res_lst, res_lst_val)